# 天气图片分类 - 智海算法调优
## 四类天气识别: sunny(晴天) / rainy(雨天) / cloudy(阴天) / snowy(雪天)

**评分规则**: 最终得分 = F1分数 × 100
**同分比较**: 模型推理时间 → 代码执行效率

### 使用步骤:
1. 插入数据集模块（左侧模块图标搜索）
2. 依次运行各Cell完成训练
3. 部署应用

## Step 0: 安装依赖（如有缺失）

In [ ]:
!pip install torch torchvision scikit-learn Pillow -q

## Step 1: 配置参数

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms, models
from sklearn.metrics import f1_score, accuracy_score
import time

# ======== 配置参数 ========
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_EPOCHS = 30          # 总训练轮数（Mo平台建议先用小值测试）
WARMUP_EPOCHS = 3        # 预热轮数（仅训练分类头）
LEARNING_RATE = 1e-3     # 分类头学习率
FINE_TUNE_LR = 1e-4      # 微调学习率
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0
NUM_CLASSES = 4
CLASS_NAMES = ["sunny", "rainy", "cloudy", "snowy"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")

## Step 2: 数据增强管线

In [ ]:
# 训练增强（含多种数据增强策略）
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.TrivialAugmentWide(),      # 自动增强
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, mode='pixel'),  # 随机擦除
])

# 验证/测试增强
val_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("数据增强管线配置完成")

## Step 3: 加载数据集
### 方式A: 使用Mo平台数据集（推荐）
在左侧模块图标中搜索天气数据集并插入，然后运行下方代码。

### 方式B: 上传自己的数据集
将数据按以下结构上传到 `./data/` 目录:
```
data/train/sunny/*.jpg
data/train/rainy/*.jpg
data/train/cloudy/*.jpg
data/train/snowy/*.jpg
data/val/sunny/*.jpg (可选)
```

In [ ]:
# ============ 数据集加载 ============
# 请根据实际情况修改 DATA_PATH

# --- 方式1: 使用Mo平台数据集模块（插入数据集模块后自动生成路径）---
# 例如: DATA_PATH = "./datasets/xxx-weather-dataset/"

# --- 方式2: 如果有上传的数据 ---
DATA_PATH = "./data/"

# --- 方式3: 使用公开天气数据集 Multi-class Weather Dataset (MWD) ---
# 可以从 Kaggle 或 Mendeley 下载: https://data.mendeley.com/datasets/4drtyfjtfy/1
# 解压后设置 DATA_PATH 指向数据集根目录


class WeatherDataset(Dataset):
    """通用天气数据集加载器"""
    def __init__(self, root_dir, transform=None, class_names=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
        
        if os.path.isdir(root_dir):
            for class_name in CLASS_NAMES:
                class_dir = os.path.join(root_dir, class_name)
                if os.path.isdir(class_dir):
                    for fname in os.listdir(class_dir):
                        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                            self.samples.append((
                                os.path.join(class_dir, fname),
                                self.class_to_idx[class_name]
                            ))
                else:
                    print(f"Warning: 未找到类别文件夹 {class_dir}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, label


# 尝试加载数据集
train_dataset = None
val_dataset = None

# 尝试按标准结构加载
train_path = os.path.join(DATA_PATH, "train")
val_path = os.path.join(DATA_PATH, "val")

if os.path.isdir(train_path):
    train_dataset = WeatherDataset(train_path, transform=train_transform)
    print(f"训练集: {len(train_dataset)} 张图片")
    
    # 统计各类别数量
    class_counts = {}
    for _, label in train_dataset.samples:
        name = CLASS_NAMES[label]
        class_counts[name] = class_counts.get(name, 0) + 1
    for name, count in class_counts.items():
        print(f"  {name}: {count} 张")

if os.path.isdir(val_path):
    val_dataset = WeatherDataset(val_path, transform=val_transform)
    print(f"验证集: {len(val_dataset)} 张图片")
elif train_dataset is not None:
    # 从训练集切分15%作为验证集
    from torch.utils.data import random_split
    val_size = max(1, int(0.15 * len(train_dataset)))
    train_size = len(train_dataset) - val_size
    train_dataset, val_dataset = random_split(
        train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42),
    )
    print(f"训练集: {train_size} 张 (从训练集切分)")
    print(f"验证集: {val_size} 张 (从训练集切分)")

if train_dataset is None:
    print("=" * 50)
    print("⚠️ 未找到数据集！请执行以下操作之一:")
    print("1. 在左侧模块图标搜索'天气'或'weather'数据集并插入")
    print("2. 上传自己的数据到 ./data/train/ 目录")
    print("3. 从 Mendeley 下载 MWD 数据集")
    print("   https://data.mendeley.com/datasets/4drtyfjtfy/1")
    print("=" * 50)

## Step 4: 数据加载器

In [ ]:
if train_dataset is not None:
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    print(f"训练批次: {len(train_loader)}")

if val_dataset is not None:
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
    )
    print(f"验证批次: {len(val_loader)}")

## Step 5: 构建 EfficientNet-B0 模型
EfficientNet-B0: 5.3M参数, 在4类天气识别任务上达到97.4%准确率

In [ ]:
def build_model(num_classes=NUM_CLASSES):
    """构建EfficientNet-B0天气分类模型"""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model.to(DEVICE)


model = build_model()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型: EfficientNet-B0")
print(f"总参数: {total_params/1e6:.2f}M | 可训练: {trainable_params/1e6:.2f}M")
print(f"类别: {CLASS_NAMES}")

## Step 6: MixUp 数据增强函数

In [ ]:
def mixup_data(x, y, alpha=0.2):
    """MixUp增强: 对两张图片及其标签进行线性插值混合"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


print("MixUp函数就绪")

## Step 7: 训练函数

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch, amp_enabled):
    """训练一个epoch"""
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []
    
    for batch_idx, (images, targets) in enumerate(loader):
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        
        # 50%概率使用MixUp
        if epoch >= WARMUP_EPOCHS and torch.rand(1).item() < 0.5:
            images, targets_a, targets_b, lam = mixup_data(images, targets, 0.2)
            mixup_active = True
        else:
            mixup_active = False
        
        optimizer.zero_grad(set_to_none=True)
        
        if amp_enabled:
            with autocast():
                outputs = model(images)
                if mixup_active:
                    loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
                else:
                    loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            if mixup_active:
                loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
            else:
                loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        
        total_loss += loss.item()
        if not mixup_active:
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    metrics = {"loss": total_loss / len(loader)}
    if all_preds:
        metrics["accuracy"] = accuracy_score(all_targets, all_preds)
        metrics["f1_macro"] = f1_score(all_targets, all_preds, average="macro")
    return metrics


@torch.no_grad()
def validate(model, loader, criterion):
    """验证"""
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    
    for images, targets in loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, targets)
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
    
    metrics = {"loss": total_loss / len(loader)}
    metrics["accuracy"] = accuracy_score(all_targets, all_preds)
    metrics["f1_macro"] = f1_score(all_targets, all_preds, average="macro")
    metrics["f1_weighted"] = f1_score(all_targets, all_preds, average="weighted")
    # 各类别F1
    per_class_f1 = f1_score(all_targets, all_preds, average=None)
    for i, name in enumerate(CLASS_NAMES):
        if i < len(per_class_f1):
            metrics[f"f1_{name}"] = per_class_f1[i]
    return metrics


print("训练函数就绪")

## Step 8: 开始训练

In [ ]:
if train_dataset is not None:
    print("=" * 60)
    print("开始训练天气分类模型")
    print(f"设备: {DEVICE} | Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE}")
    print(f"训练集: {len(train_dataset)} | 验证集: {len(val_dataset) if val_dataset else 0}")
    print("=" * 60)
    
    # 损失函数 (Label Smoothing)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    
    # 第一阶段: 冻结backbone, 仅训练分类头
    for param in model.parameters():
        param.requires_grad = True
    for name, param in model.named_parameters():
        if "classifier" not in name and "fc" not in name:
            param.requires_grad = False
    
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    best_f1 = 0.0
    start_time = time.time()
    
    for epoch in range(NUM_EPOCHS):
        # Phase 2: 解冻全模型
        if epoch == WARMUP_EPOCHS:
            print(f"\n>>> Epoch {epoch+1}: 解冻backbone, 进入全模型微调阶段")
            for param in model.parameters():
                param.requires_grad = True
            # 分组学习率
            head_params = []
            backbone_params = []
            for name, param in model.named_parameters():
                if not param.requires_grad:
                    continue
                if "classifier" in name or "fc" in name:
                    head_params.append(param)
                else:
                    backbone_params.append(param)
            optimizer = optim.AdamW([
                {"params": head_params, "lr": LEARNING_RATE},
                {"params": backbone_params, "lr": FINE_TUNE_LR},
            ], weight_decay=WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
        
        # 训练
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler,
            epoch, torch.cuda.is_available(),
        )
        scheduler.step()
        
        # 验证
        if val_loader is not None:
            val_metrics = validate(model, val_loader, criterion)
            epoch_str = (f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
                        f"Loss: {train_metrics['loss']:.4f} | "
                        f"Val Acc: {val_metrics['accuracy']:.4f} | "
                        f"Val F1: {val_metrics['f1_macro']:.4f}")
            print(epoch_str)
            
            # 保存最佳模型
            if val_metrics['f1_macro'] > best_f1:
                best_f1 = val_metrics['f1_macro']
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "metrics": val_metrics,
                    "class_names": CLASS_NAMES,
                    "image_size": IMAGE_SIZE,
                }, "best_model.pth")
                print(f"  >>> Best model saved! F1={best_f1:.4f}")
        else:
            print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {train_metrics['loss']:.4f}")
    
    elapsed = time.time() - start_time
    print(f"\n训练完成! 总耗时: {elapsed/60:.1f}分钟")
    print(f"最佳F1分数: {best_f1:.4f} → 最终得分: {best_f1*100:.1f}")
    
    # 保存最终模型 (TorchScript)
    model.eval()
    try:
        example = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
        traced = torch.jit.trace(model, example)
        traced.save("model_scripted.pt")
        print("TorchScript模型已保存: model_scripted.pt")
    except Exception as e:
        print(f"TorchScript导出跳过: {e}")
else:
    print("⚠️ 请先加载数据集再训练!")

## Step 9: 推理测试（单张图片）

In [ ]:
@torch.no_grad()
def predict_image(image_path, model_path="best_model.pth"):
    """预测单张图片的天气类别"""
    # 加载模型
    model = build_model()
    if os.path.exists(model_path):
        ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    model.eval()
    
    # 预处理
    image = Image.open(image_path).convert("RGB")
    tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    
    # 推理 + 计时
    start = time.time()
    output = model(tensor)
    probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    elapsed = (time.time() - start) * 1000
    
    pred_idx = int(np.argmax(probs))
    
    print(f"预测结果: {CLASS_NAMES[pred_idx]}")
    print(f"推理时间: {elapsed:.2f}ms")
    print("各类别置信度:")
    for i, (name, prob) in enumerate(zip(CLASS_NAMES, probs)):
        marker = " <--" if i == pred_idx else ""
        print(f"  {name}: {prob*100:5.1f}%{marker}")
    
    return CLASS_NAMES[pred_idx], probs


# 测试: 如果有测试图片可以运行
# predict_image("./test_sample.jpg")

## Step 10: Handle 函数（应用部署接口）
**重要**: 这是部署到Mo平台的核心函数，系统通过它来识别输入输出参数。

In [ ]:
def handle(image_path, model_path="best_model.pth"):
    """
    天气分类应用的handle函数
    
    输入参数:
        image_path: str  - 待分类的图片文件路径
        model_path: str  - 模型权重文件路径 (默认: best_model.pth)
    
    输出:
        dict: {
            "prediction": str,      - 预测天气类别 (sunny/rainy/cloudy/snowy)
            "confidence": float,     - 置信度 (0~1)
            "probabilities": dict,  - 各类别概率分布
            "inference_time_ms": float, - 推理时间(毫秒)
        }
    """
    import time as _time
    
    # 加载模型
    model = build_model()
    if os.path.exists(model_path):
        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    model.eval()
    
    # 预处理图片
    image = Image.open(image_path).convert("RGB")
    tensor = val_transform(image).unsqueeze(0)
    
    # 推理
    start = _time.time()
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    inference_time = (_time.time() - start) * 1000
    
    # 结果
    pred_idx = int(np.argmax(probs))
    prediction = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])
    probabilities = {name: float(p) for name, p in zip(CLASS_NAMES, probs)}
    
    result = {
        "prediction": prediction,
        "confidence": confidence,
        "probabilities": probabilities,
        "inference_time_ms": round(inference_time, 2),
    }
    
    return result


print("Handle函数已定义，可用于部署")
print("输入参数: image_path (图片路径), model_path (模型路径)")
print("输出: 天气分类预测结果")